# Building Stateful AI Workflow

In [1]:
from langgraph.graph import StateGraph

#### 1. States

In [2]:
from typing import TypedDict,Optional

class AuthState(TypedDict):
    username:Optional[str]
    password:Optional[str]
    is_authenticated:Optional[bool]
    output:Optional[str]
    
    

#### Example Objects and Their States

#### object 1: Successfull Login

In [3]:
auth_state_1: AuthState={
    'username':'ram_yadav',
    'password':'ram@123',
    'is_authenticated':True,
    'output':'login successful'
}

print(f'auth_state_1:{auth_state_1}')

auth_state_1:{'username': 'ram_yadav', 'password': 'ram@123', 'is_authenticated': True, 'output': 'login successful'}


#### object 2: Unsuccessful Login

In [4]:
auth_state_2: AuthState={
    'username':'',
    'password':'shyam@123',
    'is_authenticated':False,
    'output':'Authentication failed. Please try again'
}
print(f'auth_state_2: {auth_state_2}')

auth_state_2: {'username': '', 'password': 'shyam@123', 'is_authenticated': False, 'output': 'Authentication failed. Please try again'}


#### 2. Nodes

In [5]:
def input_node(state):
    print(state)
    
    if state.get('username','')=='':
        username=input('What is your username?')
    
    password=input('Enter your password')
    
    if state.get('username','')=='':
        return {'username':username,'password':password}
    else:
        return {'password':password}

In [6]:
input_node(auth_state_1)

{'username': 'ram_yadav', 'password': 'ram@123', 'is_authenticated': True, 'output': 'login successful'}


{'password': 'ddf'}

In [7]:
input_node(auth_state_2)

{'username': '', 'password': 'shyam@123', 'is_authenticated': False, 'output': 'Authentication failed. Please try again'}


{'username': 'dsd', 'password': ''}

#### Validate Credentials Node

In [8]:
def validate_credentials_node(state):
    username=state.get('username','')
    password=state.get('password')
    
    print(f'username:{username} , password:{password}')
    # simulated credentials validations
    if username=='test_user' and password=='secure_password':
        is_authenticated=True
    else:
        is_authenticated=False
    return{'is_authenticated':is_authenticated}

#### Incorrect Format

In [9]:
validate_credentials_node(auth_state_1)

username:ram_yadav , password:ram@123


{'is_authenticated': False}

#### Correct Format

In [10]:
auth_state_3:AuthState={
    'username':'test_user',
    'password':'secure_password',
    'is_authenticated':True,
    'output':'Authentications Failed. Please try again'
}
print(f'auth_state_3:{auth_state_3}')

auth_state_3:{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Authentications Failed. Please try again'}


In [11]:
validate_credentials_node(auth_state_3)

username:test_user , password:secure_password


{'is_authenticated': True}

#### Defining the Success Node

In [12]:
def success_node(state):
    return {'output':'Autentications successful! Welcome'}

In [13]:
success_node(auth_state_3)

{'output': 'Autentications successful! Welcome'}

#### Define failure node

In [14]:
def failure_node(state):
    return {'output':'Not Successful, please try again'}

#### Defining the router node

In [15]:
def router(state):
    if state['is_authenticated']:
        return 'success_node'
    else:
        return 'failure_node'

#### Creating the graph

In [16]:
from langgraph.graph import StateGraph
from langgraph.graph import END

In [17]:
# creating an instances of StateGraph with the GraphState structure
workflow=StateGraph(AuthState)
workflow

#### Adding nodes to the graph

In [18]:
workflow.add_node('InputNode',input_node)
workflow.add_node('ValidateCredential',validate_credentials_node)
workflow.add_node('Success',success_node)
workflow.add_node('Failure',failure_node)


#### Edges

In [19]:
workflow.add_edge("InputNode","ValidateCredential")
workflow.add_edge('Success',END)
workflow.add_edge('Failure','InputNode')

#### Building an Authentication Workflow

In [20]:
workflow.add_conditional_edges('ValidateCredential',router,{'success_node':'Success','Failure_node':'Failure'})

#### Setting the Entry Point

In [21]:
workflow.set_entry_point('InputNode')

#### Compiling the Workflow

In [22]:
app=workflow.compile()

In [23]:
inputs={"username":'test_user'}
result=app.invoke(inputs)
print(result)

{'username': 'test_user'}
username:test_user , password:secure_password
{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Autentications successful! Welcome'}


#### Defining the state for the QA workflow

In [24]:
# Define the structure of the QA state
class QAState(TypedDict):
    question:Optional[str]
    context:Optional[str]
    answer:Optional[str]

In [25]:
# create an example of object
qa_state_example=QAState(
    question='What is the purpose of this guided project',
    context="This project focuses on building a chatbot using python",
    answer=None
    
)
# Print the attributes
for key,values in qa_state_example.items():
    print(f'{key}:{values}')

question:What is the purpose of this guided project
context:This project focuses on building a chatbot using python
answer:None


#### Defining the input validations node

In [26]:
def input_validation_node(state):
    question=state.get('question','').strip()
    if not question:
        return {'valid':False,'error':"question cannot be empty"}
    return {'valid':True}

In [27]:
input_validation_node(qa_state_example)

{'valid': True}

#### Defining the context providers

In [28]:
def context_provider_node(state):
    questions=state.get('question','').lower()
    #check if the questions is related to the guided project
    if 'langgraph' in questions or 'guided project' in questions:
        context=(
            "this guided project is about langGraph, a python library to design state-based workflow"
            "Langgraph simpilifies building complex applications by connecting modular nodes with conditional edges"
        )
        return {'context':context}
    # if unrelated, set context to null
    return {'context':None}
    

#### **Integrating LLM for QA Workflow**  

<div style="text-align: center;">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Djs-AScfvwE8fnKButiPXg/Screenshot%202024-12-20%20at%204-20-07%E2%80%AFPM-mh%20-2-.png" alt="Screenshot" width="150">
</div>


In this step, we are building a node that utilizes an LLM (Large Language Model) to answer user questions based on the provided context. If the question is unrelated to the guided project, the node handles this gracefully by returning a predefined response. This approach uses `meta-llama/llama-4-maverick-17b-128e-instruct-fp8` model interface.


In [29]:
from langchain_ollama import OllamaLLM
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [30]:
def llm_qa_node(state):
    #extract the questions and context from the node
    question=state.get('question','')
    context=state.get('context',None)
    
    #check for missing context and return a fallback response
    if not context:
        return {'answer':'I dont have the enough context to answer your question.'}
    
    # construct the prompt dynamically
    prompt=f'Context:{context}\nQuestion:{question}\nAnswer the question based on the provided context'
    
    try:
        response=llm.invoke(prompt)
        return{'answer':response}
    except Exception as e:
        return {'answer':f'an error occured:{str(e)}'}

#### Creating the QA workflow graph

In [31]:
qa_workflow=StateGraph(QAState)
qa_workflow.add_node("InputNode",input_validation_node)
qa_workflow.add_node("ContextNode",context_provider_node)
qa_workflow.add_node('QANode',llm_qa_node)


#### Set entry point

In [32]:
qa_workflow.set_entry_point("InputNode")

#### Set edge point

In [33]:
qa_workflow.add_edge("InputNode","ContextNode")
qa_workflow.add_edge("ContextNode","QANode")
qa_workflow.add_edge("QANode",END)

#### compile

In [34]:
qa_app=qa_workflow.compile()

#### Ask an irrelevant question

In [35]:
qa_app.invoke({'question':'what is the weather today'})

{'question': 'what is the weather today',
 'context': None,
 'answer': 'I dont have the enough context to answer your question.'}

#### Ask relevant questions

In [36]:
qa_app.invoke({'question':'what is langgraph'})

{'question': 'what is langgraph',
 'context': 'this guided project is about langGraph, a python library to design state-based workflowLanggraph simpilifies building complex applications by connecting modular nodes with conditional edges',
 'answer': AIMessage(content='# What is LangGraph?\n\n**LangGraph** is a Python library designed for building **state-based workflows**. It simplifies the process of creating complex applications by:\n\n- **Connecting modular nodes** — allowing developers to break down functionality into reusable, modular components\n- **Using conditional edges** — enabling dynamic routing and decision-making within the workflow based on the current state\n\nIn essence, LangGraph provides a structured framework for designing workflows where different nodes (units of work) are linked together, with the flow between them determined by conditional logic based on the state of the application. This makes it easier to build sophisticated, multi-step applications in a clean 

#### drawing mermaid graph

In [38]:
print(qa_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	InputNode(InputNode)
	ContextNode(ContextNode)
	QANode(QANode)
	__end__([<p>__end__</p>]):::last
	ContextNode --> QANode;
	InputNode --> ContextNode;
	__start__ --> InputNode;
	QANode --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [37]:
import gradio as gr 

def funct(question):
    ans=qa_app.invoke({'question':question})
    return ans

demo=gr.Interface(
    fn=funct,
    inputs=gr.Textbox(label='enter some text'),
    outputs=gr.Textbox(label='output')
)
demo.launch()

c:\Users\Pappu Yadav\Desktop\Agentic-Design_pattern\Agentic_design_patterns\my_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
